/content/drive/MyDrive/research

In [1]:
%pip install thop opencv-python psutil requests matplotlib numpy huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/N /content/drive/MyDrive/research/custom_experiments/


In [2]:
# collect_per_class_metrics.py
import sys
sys.path.append('/content/drive/MyDrive/research')
from pathlib import Path
import re
import pandas as pd
from ultralytics import YOLO
import os
os.chdir('/content/drive/MyDrive/research')

ROOT = "variative_synt_experiments/"
DATASET_BASE = "TACO_5_variations"
OUT = "synthetic_per_class_analysis"

CLASS_NAMES = {
    0: "Disposable Cup",
    1: "Metal Can",
    2: "Styrofoam",
}

EXP_RE = re.compile(
    r"EX_(?P<experiment>\d+)_F(?P<finetune>[01])_A\d+_(?P<model>[NMSL])_?(?P<taco>\d+)",
    re.IGNORECASE,
)


def parse_experiment_config(folder_name: str):
    match = EXP_RE.search(folder_name)
    if not match:
        return None

    return {
        "experiment_id": int(match.group("experiment")),
        "finetuning": bool(int(match.group("finetune"))),
        "model_size": match.group("model").lower(),
        "taco_train_pct": int(match.group("taco")),
    }


def find_variation_id(var_dir: Path):
    match = re.search(r"var(\d+)_results", var_dir.name, re.IGNORECASE)
    return int(match.group(1)) if match else None


def get_data_yaml(variation_id: int, taco_pct: int) -> Path:
    return Path(DATASET_BASE) / f"var{variation_id}" / f"subset_{taco_pct}" / "data.yaml"


def find_best_weight(train_dir: Path):
    candidates = [
        train_dir / "weights" / "best.pt",
        train_dir / "best.pt",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


def extract_metrics(model_path: Path, data_yaml: Path):
    model = YOLO(str(model_path))

    metrics = model.val(
        data=str(data_yaml),
        split="val",
        verbose=False,
        plots=False,
    )

    rows = []

    for class_id, class_name in CLASS_NAMES.items():
        if hasattr(metrics.box, "class_result"):
            precision, recall, ap50, map50_95 = metrics.box.class_result(class_id)
        else:
            precision = metrics.box.p[class_id] if hasattr(metrics.box, "p") else None
            recall = metrics.box.r[class_id] if hasattr(metrics.box, "r") else None
            ap50 = metrics.box.ap50[class_id] if hasattr(metrics.box, "ap50") else None
            map50_95 = metrics.box.maps[class_id]

        rows.append({
            "class_id": class_id,
            "class_name": class_name,
            "precision": precision,
            "recall": recall,
            "AP50": ap50,
            "mAP50-95": map50_95,
        })

    return rows


def main():
    root = Path(ROOT)
    output = Path(OUT)
    output.mkdir(parents=True, exist_ok=True)

    all_rows = []

    for var_dir in sorted(root.glob("var*_results")):
        variation_id = find_variation_id(var_dir)

        if variation_id is None:
            print(f"Skipping invalid variation folder: {var_dir}")
            continue

        for model_dir in sorted(var_dir.iterdir()):
            if not model_dir.is_dir():
                continue

            for exp_dir in sorted(model_dir.iterdir()):
                if not exp_dir.is_dir():
                    continue

                config = parse_experiment_config(exp_dir.name)

                if config is None:
                    print(f"Skipping unrecognized experiment folder: {exp_dir}")
                    continue

                data_yaml = get_data_yaml(
                    variation_id=variation_id,
                    taco_pct=config["taco_train_pct"],
                )

                if not data_yaml.exists():
                    print(f"Missing data.yaml: {data_yaml}")
                    continue

                train_dirs = sorted(exp_dir.glob("train*"))

                if not train_dirs:
                    print(f"No train folder found in: {exp_dir}")
                    continue

                train_dir = train_dirs[0]
                best_pt = find_best_weight(train_dir)

                if best_pt is None:
                    print(f"No best.pt found in: {train_dir}")
                    continue

                print(f"Processing: {best_pt}")
                print(f"Using data: {data_yaml}")

                per_class_rows = extract_metrics(best_pt, data_yaml)

                for row in per_class_rows:
                    all_rows.append({
                        "variation_id": variation_id,
                        "data_yaml": str(data_yaml),
                        "experiment_folder": exp_dir.name,
                        **config,
                        **row,
                    })

    raw_df = pd.DataFrame(all_rows)

    if raw_df.empty:
        print("No metrics collected.")
        return

    raw_path = output / "per_class_metrics_raw.csv"
    raw_df.to_csv(raw_path, index=False)
    print(f"Saved raw metrics: {raw_path}")

    group_cols = [
        "experiment_id",
        "finetuning",
        "model_size",
        "taco_train_pct",
        "class_id",
        "class_name",
    ]

    metric_cols = ["precision", "recall", "AP50", "mAP50-95"]

    avg_df = (
        raw_df
        .groupby(group_cols, as_index=False)[metric_cols]
        .mean()
        .sort_values(
            by=["taco_train_pct", "model_size", "finetuning", "class_id"]
        )
    )

    avg_path = output / "per_class_metrics_averaged_all.csv"
    avg_df.to_csv(avg_path, index=False)
    print(f"Saved averaged metrics: {avg_path}")

    for taco_pct, taco_df in avg_df.groupby("taco_train_pct"):
        out_path = output / f"per_class_metrics_taco_{taco_pct}.csv"
        taco_df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")


if __name__ == "__main__":
    main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
Processing: variative_synt_experiments/var1_results/L/EX_9_F1_A1_L_100/train/weights/best.pt
Using data: TACO_5_variations/var1/subset_100/data.yaml
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


100%|██████████| 755k/755k [00:00<00:00, 129MB/s]
val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var1/subset_100/labels/val.cache... 40 images, 0 backgrounds, 0 corrupt: 100%|██████████| 40/40 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:21<00:00,  7.18s/it]

                   all         40         51      0.576      0.648      0.571      0.478
Speed: 0.1ms preprocess, 10.2ms inference, 0.0ms loss, 6.5ms postprocess per image


Processing: variative_synt_experiments/var1_results/L/EX_9_F1_A1_L_30/train/weights/best.pt
Using data: TACO_5_variations/var1/subset_30/data.yaml
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var1/subset_30/labels/val.cache... 12 images, 0 backgrounds, 0 corrupt: 100%|██████████| 12/12 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:10<00:00, 10.91s/it]

...

                   all         20         25      0.827       0.62      0.617       0.51
Speed: 0.1ms preprocess, 5.5ms inference, 0.0ms loss, 1.1ms postprocess per image


Processing: variative_synt_experiments/var3_results/S/EX_9_F1_A1_S_80/train/weights/best.pt
Using data: TACO_5_variations/var3/subset_80/data.yaml
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv12s summary (fused): 376 layers, 9,075,369 parameters, 0 gradients, 19.3 GFLOPs


val: Scanning /content/drive/MyDrive/research/TACO_5_variations/var3/subset_80/labels/val.cache... 32 images, 0 backgrounds, 0 corrupt: 100%|██████████| 32/32 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:01<00:00,  1.02it/s]

                   all         32         68      0.811      0.637      0.756      0.632
Speed: 0.1ms preprocess, 2.6ms inference, 0.0ms loss, 0.9ms postprocess per image


Saved raw metrics: synthetic_per_class_analysis/per_class_metrics_raw.csv
Saved averaged metrics: synthetic_per_class_analysis/per_class_metrics_averaged_all.csv
Saved: synthetic_per_class_analysis/per_class_metrics_taco_30.csv
Saved: synthetic_per_class_analysis/per_class_metrics_taco_50.csv
Saved: synthetic_per_class_analysis/per_class_metrics_taco_80.csv
Saved: synthetic_per_class_analysis/per_class_metrics_taco_100.csv


In [ ]:
!cp -r /content/variative_cust_experiments/var1_results/L/ /content/drive/MyDrive/research/variative_cust_experiments/var1_results/L/

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/research')

from ultralytics import YOLO
import os
os.chdir('/content/drive/MyDrive/research')
#from drive.MyDrive.research.ultralytics import YOLO

# ---------------- CONFIG ----------------
DATA_YAML_100 = "TACO_splits_with_test/Custom_TACO_split_100/data.yaml"
DATA_YAML_80 = "TACO_splits_with_test/Custom_TACO_split_80/data.yaml"
DATA_YAML_50 = "TACO_splits_with_test/Custom_TACO_split_50/data.yaml"
DATA_YAML_30 = "TACO_splits_with_test/Custom_TACO_split_30/data.yaml"

DATA = [DATA_YAML_100, DATA_YAML_80, DATA_YAML_50, DATA_YAML_30]
DATA_NOTATIONS = ["100", "80", "50", "30"]

PRETRAINED_MODEL_10 = "experiments/models/custom/l_10.pt"
PRETRAINED_MODEL_30 = "experiments/models/custom/l_30.pt"
PRETRAINED_MODEL_50 = "experiments/models/custom/l_50.pt"
PRETRAINED_MODEL_80 = "experiments/models/custom/l_80.pt"
MDL = "L"
EXP = "5"
PATIENCE = 100
EPOCHS = 250
BATCH_SIZE = 64
WORKERS = 4
LR = 0.001
FREEZE_BACKBONE = True
# ----------------------------------------
def main():
    # Load pretrained YOLO model
    model10 = YOLO(PRETRAINED_MODEL_10)
    model30 = YOLO(PRETRAINED_MODEL_30)
    model50 = YOLO(PRETRAINED_MODEL_50)
    model80 = YOLO(PRETRAINED_MODEL_80)
    #---------------------EX5_F1
    for i, dataset in enumerate(DATA):
      results10 = model10.train(
          data=dataset,
          epochs=EPOCHS,
          batch=BATCH_SIZE,
          imgsz=640,          # input image size
          project="/content/"+MDL+"/EX_"+EXP+"_F1_A1_"+MDL+"10_"+DATA_NOTATIONS[i],
          amp=True,
          freeze=True,
          workers=WORKERS,
          lr0=LR,
          patience=PATIENCE
      )
      results30 = model30.train(
          data=dataset,
          epochs=EPOCHS,
          batch=BATCH_SIZE,
          imgsz=640,          # input image size
          project="/content/"+MDL+"/EX_"+EXP+"_F1_A1_"+MDL+"30_"+DATA_NOTATIONS[i],
          amp=True,
          freeze=True,
          workers=WORKERS,
          lr0=LR,
          patience=PATIENCE
      )
      results50 = model50.train(
          data=dataset,
          epochs=EPOCHS,
          batch=BATCH_SIZE,
          imgsz=640,          # input image size
          project="/content/"+MDL+"/EX_"+EXP+"_F1_A1_"+MDL+"50_"+DATA_NOTATIONS[i], # Changed to local path to avoid Google Drive write issues
          amp=True,
          freeze=True,
          workers=WORKERS,
          lr0=LR,
          patience=PATIENCE
      )

      results80 = model80.train(
          data=dataset,
          epochs=EPOCHS,
          batch=BATCH_SIZE,
          imgsz=640,          # input image size
          project="/content/"+MDL+"/EX_"+EXP+"_F1_A1_"+MDL+"80_"+DATA_NOTATIONS[i],
          amp=True,
          freeze=True,
          workers=WORKERS,
          lr0=LR,
          patience=PATIENCE
      )


    print("Training complete! Model weights saved locally in /content/yolo_runs. Remember to copy them to your Google Drive.")
    # Example of how to copy the results from results5 to Google Drive after training:
    # !mkdir -p /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_80
    # !cp -r /content/yolo_runs/M/EX_2_F0_A1_M_80/train2 /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_80/
    #
    # Example of how to copy the results from results6 to Google Drive after training:
    # !mkdir -p /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_100
    # !cp -r /content/yolo_runs/M/EX_2_F0_A1_M_100/train2 /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_100/

if __name__ == '__main__':
    #import multiprocessing
    #multiprocessing.freeze_support()
    main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: task=detect, mode=train, model=experiments/models/custom/l_10.pt, data=TACO_splits_with_test/Custom_TACO_split_100/data.yaml, epochs=250, time=None, patience=100, batch=64, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=4, project=/content/L/EX_5_F1_A1_L10_100, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True

100%|██████████| 755k/755k [00:00<00:00, 42.2MB/s]


Overriding model.yaml nc=80 with nc=3

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     37120  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2, 1, 2]         
  2                  -1  2    173824  ultralytics.nn.modules.block.C3k2            [128, 256, 2, True, 0.25]     
  3                  -1  1    147968  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2, 1, 4]        
  4                  -1  2    691712  ultralytics.nn.modules.block.C3k2            [256, 512, 2, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  4   4540416  ultralytics.nn.modules.block.A2C2f           [512, 512, 4, True, 4, True, 1.5]
  7                  -1  1   2360320  ultralyt

train: Scanning /content/drive/MyDrive/research/TACO_splits_with_test/Custom_TACO_split_100/labels/train.cache... 120 images, 0 backgrounds, 0 corrupt: 100%|██████████| 120/120 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


...

    119/250      40.4G     0.4552     0.3926      0.916         88        640: 100%|██████████| 1/1 [00:00<00:00,  1.40it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  8.85it/s]

                   all         12         13      0.643      0.445      0.427      0.336
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 19, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



119 epochs completed in 0.098 hours.
Optimizer stripped from /content/L/EX_5_F1_A1_L80_30/train/weights/last.pt, 53.7MB
Optimizer stripped from /content/L/EX_5_F1_A1_L80_30/train/weights/best.pt, 53.7MB

Validating /content/L/EX_5_F1_A1_L80_30/train/weights/best.pt...
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12l summary (fused): 674 layers, 26,395,865 parameters, 0 gradients, 82.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  9.36it/s]


                   all         12         13      0.854      0.914      0.891      0.851
        Disposable Cup          4          5          1      0.993      0.995      0.975
             Metal Can          4          4      0.819          1      0.995      0.895
             Styrofoam          4          4      0.742       0.75      0.684      0.684
Speed: 0.1ms preprocess, 4.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/L/EX_5_F1_A1_L80_30/train
Training complete! Model weights saved locally in /content/yolo_runs. Remember to copy them to your Google Drive.


In [ ]:
!cp -r /content/L /content/drive/MyDrive/research/custom_experiments/L/

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/research')

from ultralytics import YOLO
import os
os.chdir('/content/drive/MyDrive/research')
#from drive.MyDrive.research.ultralytics import YOLO

# ---------------- CONFIG ----------------
DATA_YAML_100 = "TACO_splits/split_100/data.yaml"
DATA_YAML_90 = "TACO_splits/split_90/data.yaml"
DATA_YAML_80 = "TACO_splits/split_80/data.yaml"
DATA_YAML_70 = "TACO_splits/split_70/data.yaml"
DATA_YAML_60 = "TACO_splits/split_60/data.yaml"
DATA_YAML_50 = "TACO_splits/split_50/data.yaml"
DATA_YAML_40 = "TACO_splits/split_40/data.yaml"
DATA_YAML_30 = "TACO_splits/split_30/data.yaml"
PRETRAINED_MODEL_0 = "yolov12x.yaml"
PRETRAINED_MODEL_100_N = "experiments/models/official/yolo12x.pt"
EPOCHS = 1
BATCH_SIZE = 48
WORKERS = 4
LR = 0.001
FREEZE_BACKBONE = True
# ----------------------------------------
def main():
    # Load pretrained YOLO model
    model = YOLO(PRETRAINED_MODEL_0)
    model2 = YOLO(PRETRAINED_MODEL_100_N)

    # Train NO AUG
    # Train AUG EX2
    # Train AUG EX2
    results = model.train(
        data=DATA_YAML_100,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=640,          # input image size
        project="/content/X/EX_1_F0_A0_X_100",
        amp=True,
        #freeze=True,
        workers=WORKERS,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        translate=0.0,
        scale=0.0,
        fliplr=0.0,
        mosaic=0.0,
        erasing=0.0,
        auto_augment=None
    )
    results2 = model.train(
        data=DATA_YAML_100,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=640,          # input image size
        project="/content/X/EX_1_F0_A1_X_100",
        amp=True,
        #freeze=False,
        workers=WORKERS,
        lr0=LR
    )

    print("Training complete! Model weights saved locally in /content/yolo_runs. Remember to copy them to your Google Drive.")
    # Example of how to copy the results from results5 to Google Drive after training:
    # !mkdir -p /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_80
    # !cp -r /content/yolo_runs/M/EX_2_F0_A1_M_80/train2 /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_80/
    #
    # Example of how to copy the results from results6 to Google Drive after training:
    # !mkdir -p /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_100
    # !cp -r /content/yolo_runs/M/EX_2_F0_A1_M_100/train2 /content/drive/MyDrive/research/experiments/M/EX_2_F0_A1_M_100/

if __name__ == '__main__':
    #import multiprocessing
    #multiprocessing.freeze_support()
    main()

New https://pypi.org/project/ultralytics/8.4.6 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: task=detect, mode=train, model=yolov12x.yaml, data=TACO_splits/split_100/data.yaml, epochs=1, time=None, patience=100, batch=48, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=4, project=/content/X/EX_1_F0_A0_X_100, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=

train: Scanning /content/drive/MyDrive/research/TACO_splits/split_100/labels/train.cache... 160 images, 0 backgrounds, 0 corrupt: 100%|██████████| 160/160 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



/content/drive/MyDrive/research/ultralytics/data/augment.py:1853: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /content/drive/MyDrive/research/TACO_splits/split_100/labels/val.cache... 41 images, 0 backgrounds, 0 corrupt: 100%|██████████| 41/41 [00:00<?, ?it/s]


Plotting labels to /content/X/EX_1_F0_A0_X_100/train3/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 221 weight(decay=0.0), 230 weight(decay=0.000375), 227 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /content/X/EX_1_F0_A0_X_100/train3
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

Plotting labels to /content/X/EX_1_F0_A1_X_100/train3/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 221 weight(decay=0.0), 230 weight(decay=0.000375), 227 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /content/X/EX_1_F0_A1_X_100/train3
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/1      79.9G      3.584      6.627      4.598         21        640: 100%|██████████| 4/4 [00:05<00:00,  1.26s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]


                   all         41         84   0.000456     0.0855    0.00033   0.000153

1 epochs completed in 0.007 hours.
Optimizer stripped from /content/X/EX_1_F0_A1_X_100/train3/weights/last.pt, 119.5MB
Optimizer stripped from /content/X/EX_1_F0_A1_X_100/train3/weights/best.pt, 119.5MB

Validating /content/X/EX_1_F0_A1_X_100/train3/weights/best.pt...
Ultralytics 8.3.63 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
YOLOv12x summary (fused): 674 layers, 59,248,345 parameters, 0 gradients, 184.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]


                   all         41         84   0.000456     0.0855    0.00033   0.000153
        Disposable Cup         16         20          0          0          0          0
             Metal Can         15         47   0.000179     0.0213   9.25e-05   1.85e-05
             Styrofoam         15         17    0.00119      0.235   0.000898   0.000442
Speed: 0.1ms preprocess, 7.0ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/X/EX_1_F0_A1_X_100/train3
Training complete! Model weights saved locally in /content/yolo_runs. Remember to copy them to your Google Drive.
